# Statistical full held-out segment evaluation

This notebook evaluates each frozen statistical baseline on every labelled row from the full observation whose `segment_index` belongs to the corrected test split. It does not train models, calculate channel features, choose a candidate, or choose a threshold. The three model bundles already contain their fitted preprocessing pipelines, while each candidate manifest is the authority for its validation-selected threshold.

The computation is CPU-only. This is an offline held-out-segment evaluation using stored metadata features, not a new real-time feature-extraction benchmark and not an independent-observation transfer test.

In [1]:
from pathlib import Path
from copy import deepcopy
import json
import subprocess
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit
from tqdm import tqdm

REPO_ROOT = next((parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (parent / 'src' / 'rfimt').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError('Open this notebook from a directory inside the rfimt repository.')
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from rfimt.experiments import load_experiment_spec, make_run_manifest, write_run_manifest
from rfimt.metrics import eval_binary

CONFIG_PATH = REPO_ROOT / 'configs/experiments/b0531_statistical_full_heldout_segments_v1.json'
spec = load_experiment_spec(CONFIG_PATH)
if spec['evaluation']['device'] != 'cpu':
    raise ValueError('Statistical held-out evaluation is intentionally configured for CPU.')
FEATURES = spec['representation']['features']
GROUP_COLUMN = spec['split']['group_column']
RUN_DIR = Path(spec['outputs']['run_directory'])
if RUN_DIR.exists():
    raise FileExistsError(f'Refusing to overwrite existing run directory: {RUN_DIR}')
RUN_DIR.mkdir(parents=True)
print(f'Repository: {REPO_ROOT}')
print('Device: CPU')

Repository: /u/akazantsev/ml_dl_for_rfi_mitigation
Device: CPU


## Recover complete held-out segments

Subset split indices are valid only inside the corrected subset metadata. They identify the held-out `segment_index` values; those group identifiers then select all corresponding rows from the complete metadata table.

In [2]:
group_dir = Path(spec['dataset']['group_dataset_dir'])
subset_meta = pd.read_csv(group_dir / 'subset_channels_meta.csv').fillna('None')
with np.load(spec['split']['indices_path']) as split_file:
    split_indices = {name: np.asarray(split_file[name], dtype=int) for name in ('train', 'val', 'test')}
split_groups = {name: set(subset_meta.iloc[indices][GROUP_COLUMN].tolist()) for name, indices in split_indices.items()}
if split_groups['train'] & split_groups['val'] or split_groups['train'] & split_groups['test'] or split_groups['val'] & split_groups['test']:
    raise RuntimeError('Corrected split contains overlapping segment_index values.')
test_groups = split_groups['test']
if not test_groups:
    raise RuntimeError('The corrected split has no test segments.')

full_meta = pd.read_csv(spec['dataset']['full_metadata_path']).fillna('None').copy()
evaluation_meta = full_meta.loc[full_meta[GROUP_COLUMN].isin(test_groups)].copy()
if evaluation_meta.empty or set(evaluation_meta[GROUP_COLUMN]) != test_groups:
    raise RuntimeError('The full metadata does not contain every held-out test segment.')
missing_features = sorted(set(FEATURES) - set(evaluation_meta.columns))
if missing_features:
    raise ValueError(f'Full metadata is missing required features: {missing_features}')
allowed_labels = {spec['labels']['positive'], spec['labels']['ordinary_negative'], spec['labels']['hard_negative']}
unknown_labels = sorted(set(evaluation_meta['label']) - allowed_labels)
if unknown_labels:
    raise ValueError(f'Unexpected labels in held-out full rows: {unknown_labels}')
evaluation_meta['target'] = evaluation_meta['label'].eq(spec['labels']['positive']).astype(int)
print(f'Held-out segments: {len(test_groups)}')
print(evaluation_meta['label'].value_counts().rename('rows'))

<ipython-input-2-a03c1cbd5876>:12: DtypeWarning: Columns (21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  full_meta = pd.read_csv(spec['dataset']['full_metadata_path']).fillna('None').copy()


Held-out segments: 1242
label
None          204288
NoneWNBRFI    105694
NBRFI           7970
Name: rows, dtype: int64


## CPU batch prediction and audit helpers

Features are already stored in metadata. Prediction is still batched so the notebook exposes progress and avoids a single oversized `predict_proba` call on the full held-out table. The notebook deliberately writes compact audit tables rather than three redundant full-row score files.

In [3]:
def positive_scores(model, features):
    classes = np.asarray(model.classes_)
    if not np.array_equal(classes, np.array([0, 1])):
        raise ValueError(f'Expected binary classes [0, 1], got {classes.tolist()}.')
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(features)[:, 1], 'predict_proba'
    if hasattr(model, 'decision_function'):
        return expit(model.decision_function(features)), 'decision_function_sigmoid'
    raise TypeError('Statistical candidate exposes neither predict_proba nor decision_function.')

def predict_in_batches(model, feature_frame, batch_size=100_000, description='Predicting'):
    chunks = []
    for start in tqdm(range(0, len(feature_frame), batch_size), desc=description, mininterval=5.0, miniters=10):
        scores, score_source = positive_scores(model, feature_frame.iloc[start:start + batch_size])
        chunks.append(scores)
    return np.concatenate(chunks), score_source

def count_errors(frame):
    target = frame['target'].to_numpy(dtype=int)
    prediction = frame['prediction'].to_numpy(dtype=int)
    return {
        'n_rows': int(len(frame)),
        'TN': int(((target == 0) & (prediction == 0)).sum()),
        'FP': int(((target == 0) & (prediction == 1)).sum()),
        'FN': int(((target == 1) & (prediction == 0)).sum()),
        'TP': int(((target == 1) & (prediction == 1)).sum()),
    }

def build_audit_tables(frame, candidate, threshold):
    label_rows = []
    for label, label_frame in frame.groupby('label', sort=True):
        row = count_errors(label_frame)
        row.update({'candidate': candidate, 'label': label, 'mean_score': float(label_frame['score'].mean()), 'positive_prediction_fraction': float(label_frame['prediction'].mean())})
        negatives = row['TN'] + row['FP']
        positives = row['FN'] + row['TP']
        row['false_positive_rate'] = float(row['FP'] / negatives) if negatives else None
        row['false_negative_rate'] = float(row['FN'] / positives) if positives else None
        label_rows.append(row)

    segment_rows = []
    groups = frame.groupby(GROUP_COLUMN, sort=True)
    for segment_index, segment_frame in tqdm(groups, total=frame[GROUP_COLUMN].nunique(), desc=f'{candidate}: segment audit', leave=False, mininterval=5.0, miniters=25):
        row = count_errors(segment_frame)
        row.update({GROUP_COLUMN: segment_index, 'candidate': candidate, 'labels_present': '|'.join(sorted(segment_frame['label'].unique())), 'mean_score': float(segment_frame['score'].mean()), 'positive_prediction_fraction': float(segment_frame['prediction'].mean()), 'threshold': threshold})
        segment_rows.append(row)
    return pd.DataFrame(label_rows), pd.DataFrame(segment_rows)

def json_ready(value):
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value

## Evaluate frozen candidates

A candidate's validation-selected threshold is reused exactly. The test metrics are therefore comparable to the source baseline, while `NoneWNBRFI` remains visible separately as a hard-negative label in the full held-out segments.

In [4]:
source_run_dir = Path(spec['model']['source_run_directory'])
feature_frame = evaluation_meta[FEATURES]
metrics_rows = []
candidate_frames = {}

for candidate in tqdm(spec['model']['candidates'], desc='Statistical candidates', mininterval=5.0):
    candidate_dir = source_run_dir / candidate
    bundle_path = candidate_dir / 'model.joblib'
    manifest_path = candidate_dir / 'manifest.json'
    if not bundle_path.is_file() or not manifest_path.is_file():
        raise FileNotFoundError(f'Run the statistical baseline first: missing artifacts for {candidate}.')
    bundle = joblib.load(bundle_path)
    with manifest_path.open('r', encoding='utf-8') as handle:
        source_manifest = json.load(handle)
    source_spec = source_manifest['spec']
    if source_spec['representation']['features'] != FEATURES:
        raise ValueError(f'{candidate} was trained with a different feature contract.')
    if source_spec['selection']['threshold_source'] != 'validation':
        raise ValueError(f'{candidate} threshold was not selected from validation data.')
    threshold = float(source_manifest['metrics']['validation']['threshold'])
    if not np.isclose(threshold, float(bundle['threshold'])):
        raise ValueError(f'{candidate} bundle and manifest disagree on threshold.')
    if bundle['features'] != FEATURES:
        raise ValueError(f'{candidate} bundle and evaluator disagree on feature order.')

    scores, score_source = predict_in_batches(bundle['model'], feature_frame, description=f'{candidate}: CPU inference')
    candidate_frame = evaluation_meta[[GROUP_COLUMN, 'label', 'target']].copy()
    candidate_frame['source_row_index'] = evaluation_meta.index.to_numpy(dtype=int)
    candidate_frame['score'] = scores
    candidate_frame['prediction'] = (scores >= threshold).astype(int)
    row_metrics = eval_binary(candidate_frame['target'], scores, threshold=threshold)
    if score_source == 'decision_function_sigmoid':
        row_metrics['logloss'] = None
    label_table, segment_table = build_audit_tables(candidate_frame, candidate, threshold)
    output_dir = RUN_DIR / candidate
    output_dir.mkdir()
    label_table.to_csv(output_dir / 'label_breakdown.csv', index=False)
    segment_table.to_csv(output_dir / 'full_heldout_segment_metrics.csv', index=False)

    metrics_rows.append({'candidate': candidate, 'score_source': score_source, 'threshold': threshold, **row_metrics})
    candidate_frames[candidate] = (candidate_frame, segment_table, source_manifest, bundle_path, manifest_path, score_source)

summary = pd.DataFrame(metrics_rows).sort_values(['f1', 'pr_auc'], ascending=False).reset_index(drop=True)
summary.to_csv(RUN_DIR / 'summary_metrics.csv', index=False)
summary

SGD_LogReg: CPU inference: 100%|██████████| 4/4 [00:00<00:00, 50.15it/s]

SGD_LogReg: segment audit:   0%|          | 0/1242 [00:00<?, ?it/s]
                                                                   
SGD_LinearSVM: CPU inference: 100%|██████████| 4/4 [00:00<00:00, 65.05it/s]

SGD_LinearSVM: segment audit:   0%|          | 0/1242 [00:00<?, ?it/s]
                                                                      
Poly2_LogReg: CPU inference: 100%|██████████| 4/4 [00:00<00:00,  5.04it/s]

Poly2_LogReg: segment audit:   0%|          | 0/1242 [00:00<?, ?it/s]
                                                                     
RBFapprox_LogReg: CPU inference: 100%|██████████| 4/4 [00:01<00:00,  2.41it/s]

HistGB: CPU inference: 100%|██████████| 4/4 [00:00<00:00,  4.77it/s]

HistGB: segment audit:   0%|          | 0/1242 [00:00<?, ?it/s]
                                                               
MLP: CPU inference: 100%|██████████| 4/4 [00:00<00:00, 20.37it/s]

Statistica

,candidate,score_source,threshold,TN,FP,FN,TP,accuracy,precision,recall,f1,roc_auc,pr_auc,logloss
0,SGD_LogReg,predict_proba,0.02,284277,25705,919,7051,0.916264,0.215258,0.884693,0.346265,0.913796,0.202976,2.964159
1,SGD_LinearSVM,decision_function_sigmoid,0.84,267122,42860,1056,6914,0.861879,0.138908,0.867503,0.239471,0.870622,0.127448,NaN
2,MLP,predict_proba,0.52,255860,54122,432,7538,0.828421,0.122251,0.945797,0.216516,0.964905,0.621256,0.770936
3,Poly2_LogReg,predict_proba,0.01,255987,53995,873,7097,0.827433,0.116169,0.890464,0.205525,0.858138,0.106190,6.219943
4,HistGB,predict_proba,0.18,212272,97710,76,7894,0.692450,0.074751,0.990464,0.139011,0.934329,0.177786,1.910959
5,RBFapprox_LogReg,predict_proba,0.23,218,309764,5,7965,0.025737,0.025069,0.999373,0.048910,0.499394,0.025221,0.667781


## Visual review of the largest errors

Only selected error cases load the raw channel series. This keeps the statistical evaluation focused on metadata inference while still providing the same visual evidence needed to diagnose false-positive and false-negative segments.

In [5]:
full_array = None
n_review = int(spec['evaluation']['review_segments_per_error_type'])
for candidate, (candidate_frame, segment_table, source_manifest, bundle_path, manifest_path, score_source) in tqdm(candidate_frames.items(), desc='Writing candidate diagnostics', mininterval=5.0):
    fp_queue = segment_table.loc[segment_table['FP'] > 0].nlargest(n_review, 'FP').assign(review_reason='largest_false_positive_count')
    fn_queue = segment_table.loc[segment_table['FN'] > 0].nlargest(n_review, 'FN').assign(review_reason='largest_false_negative_count')
    review_queue = pd.concat([fp_queue, fn_queue], ignore_index=True).drop_duplicates(subset=[GROUP_COLUMN], keep='first')
    output_dir = RUN_DIR / candidate
    review_queue.to_csv(output_dir / 'visual_review_queue.csv', index=False)
    diagnostic_dir = output_dir / 'diagnostics'
    diagnostic_dir.mkdir()
    if full_array is None:
        full_array = np.load(spec['dataset']['full_array_path'], mmap_mode='r')
        if len(full_array) != len(full_meta):
            raise ValueError('Full metadata and full array have different row counts.')
    for segment_index in tqdm(review_queue[GROUP_COLUMN], desc=f'{candidate}: diagnostics', leave=False, mininterval=5.0):
        source_rows = full_meta.loc[full_meta[GROUP_COLUMN].eq(segment_index)].copy()
        source_rows['source_row_index'] = source_rows.index.to_numpy(dtype=int)
        sort_column = 'channel_index' if 'channel_index' in source_rows.columns else 'source_row_index'
        source_rows = source_rows.sort_values(sort_column)
        segment_scores = candidate_frame.set_index('source_row_index').loc[source_rows.index, ['score', 'target', 'prediction']].reset_index(drop=True)
        if len(source_rows) != len(segment_scores):
            raise RuntimeError(f'{candidate}: score rows do not match diagnostic rows for segment {segment_index}.')
        profiles = np.asarray(full_array[source_rows['source_row_index'].to_numpy(dtype=int)])
        fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(11, 5), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
        image = ax0.imshow(profiles, aspect='auto', origin='lower', interpolation='nearest')
        ax0.set(ylabel='channel row', title=f'{candidate}: held-out segment {segment_index}')
        fig.colorbar(image, ax=ax0, label='recorded value')
        ax1.plot(segment_scores['score'].to_numpy(), label='model score', linewidth=1.2)
        ax1.step(np.arange(len(segment_scores)), segment_scores['target'].to_numpy(), where='mid', label='NBRFI label', alpha=0.8)
        ax1.step(np.arange(len(segment_scores)), segment_scores['prediction'].to_numpy(), where='mid', label='prediction', alpha=0.8)
        threshold = float(source_manifest['metrics']['validation']['threshold'])
        ax1.axhline(threshold, color='black', linestyle='--', linewidth=1, label=f'threshold {threshold:.2f}')
        ax1.set(xlabel='ordered channel row', ylabel='score / class', ylim=(-0.05, 1.05))
        ax1.legend(loc='upper right', ncol=2, fontsize=8)
        fig.tight_layout()
        fig.savefig(diagnostic_dir / f'segment_{segment_index}.png', dpi=150)
        plt.close(fig)

Writing candidate diagnostics: 100%|██████████| 6/6 [00:37<00:00,  6.28s/it]


## Record candidate-specific evaluation manifests

Each full held-out evaluation gets a separate manifest so later comparisons cannot confuse the candidate, threshold, or source model bundle.

In [6]:
code_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
for candidate, (candidate_frame, segment_table, source_manifest, bundle_path, manifest_path, score_source) in candidate_frames.items():
    candidate_spec = deepcopy(spec)
    candidate_spec['experiment_id'] = f"{spec['experiment_id']}__{candidate.lower()}"
    candidate_spec['model']['candidate'] = candidate
    candidate_spec['model']['score_source'] = score_source
    candidate_spec['outputs']['run_directory'] = str(RUN_DIR / candidate)
    threshold = float(source_manifest['metrics']['validation']['threshold'])
    candidate_scores = candidate_frame['score'].to_numpy()
    candidate_metrics = eval_binary(candidate_frame['target'], candidate_scores, threshold=threshold)
    if score_source == 'decision_function_sigmoid':
        candidate_metrics['logloss'] = None
    manifest = make_run_manifest(
        candidate_spec,
        code_revision=code_revision,
        metrics=json_ready({'full_heldout_rows': candidate_metrics, 'n_heldout_segments': int(len(test_groups)), 'frozen_validation_threshold': threshold, 'source_baseline_experiment_id': source_manifest['experiment_id']}),
        artifacts={
            'source_baseline_manifest': str(manifest_path),
            'source_model_bundle': str(bundle_path),
            'label_breakdown': str(RUN_DIR / candidate / 'label_breakdown.csv'),
            'segment_metrics': str(RUN_DIR / candidate / 'full_heldout_segment_metrics.csv'),
            'visual_review_queue': str(RUN_DIR / candidate / 'visual_review_queue.csv'),
            'diagnostic_directory': str(RUN_DIR / candidate / 'diagnostics'),
        },
        notes=['All rows of group-held-out segments were evaluated.', 'Threshold was reused from the source validation universe and was not selected here.', 'This is CPU metadata inference, not a feature-extraction throughput measurement or independent-observation transfer test.'] + (['SGD_LinearSVM uses sigmoid-mapped decision scores for ranking and threshold selection; logloss is intentionally not reported because these are not calibrated probabilities.'] if score_source == 'decision_function_sigmoid' else []),
    )
    write_run_manifest(RUN_DIR / candidate / 'run_manifest.json', manifest)

print(f'Wrote statistical full held-out evaluation to {RUN_DIR}')

Wrote statistical full held-out evaluation to /hercules/results/akazantsev/rfim_dataset/runs/b0531_statistical_full_heldout_segments_v1
